# Airbnb Pricing & Market Intelligence

## Data Cleaning

This notebook focuses on preparing the datasets for analysis by addressing data quality issues identified during the audit process.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

listings = pd.read_csv(
    "../data/raw/Listings.csv",
    encoding="latin1",
    low_memory=False
)

reviews = pd.read_csv(
    "../data/raw/Reviews.csv",
    encoding="latin1",
    low_memory=False
)

print("Listings Shape :", listings.shape)
print("Reviews Shape  :", reviews.shape)

Listings Shape : (279712, 33)
Reviews Shape  : (5373143, 4)


### Cleaning Inventory

This section reviews the data quality issues identified during the audit and establishes the scope of cleaning activities.

In [2]:
cleaning_inventory = pd.DataFrame({
    "Issue Category": [
        "Missing Values",
        "Duplicate Keys",
        "Data Types",
        "Domain Issues",
        "Outliers"
    ],
    "Affected Columns": [
        "district, host_response_time, host_response_rate, host_acceptance_rate, review score columns, bedrooms",
        "review_id",
        "host_since, date",
        "price, accommodates",
        "price, bedrooms, minimum_nights, maximum_nights"
    ]
})

print(cleaning_inventory.to_string(index=False))

Issue Category                                                                                       Affected Columns
Missing Values district, host_response_time, host_response_rate, host_acceptance_rate, review score columns, bedrooms
Duplicate Keys                                                                                              review_id
    Data Types                                                                                       host_since, date
 Domain Issues                                                                                    price, accommodates
      Outliers                                                        price, bedrooms, minimum_nights, maximum_nights


### Missing Value Assessment

In [3]:
missing_summary = (
    listings.isna()
            .sum()
            .reset_index()
)

missing_summary.columns = ["column", "missing_count"]

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(listings) * 100
).round(2)

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values(
    by="missing_count",
    ascending=False
)

print(missing_summary.to_string(index=False))

                     column  missing_count  missing_percent
                   district         242700            86.77
         host_response_time         128782            46.04
         host_response_rate         128782            46.04
       host_acceptance_rate         113087            40.43
        review_scores_value          91785            32.81
     review_scores_location          91775            32.81
      review_scores_checkin          91771            32.81
     review_scores_accuracy          91713            32.79
review_scores_communication          91687            32.78
  review_scores_cleanliness          91665            32.77
       review_scores_rating          91405            32.68
                   bedrooms          29435            10.52
              host_location            840             0.30
                       name            175             0.06
                 host_since            165             0.06
     host_identity_verified            1

In [4]:
# District Analysis

print("Unique District Values:")
print(listings["district"].dropna().unique())

print("\n")

print("Listings With District:")
print(listings["district"].notna().sum())

print("\n")

print("Listings Without District:")
print(listings["district"].isna().sum())

Unique District Values:
<ArrowStringArray>
['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
Length: 5, dtype: str


Listings With District:
37012


Listings Without District:
242700


In [5]:
# District by City

(
    listings.groupby("city")["district"]
            .apply(lambda x: x.notna().sum())
            .sort_values(ascending=False)
)

city
New York          37012
Bangkok               0
Cape Town             0
Hong Kong             0
Istanbul              0
Mexico City           0
Paris                 0
Rio de Janeiro        0
Rome                  0
Sydney                0
Name: district, dtype: int64

In [6]:
# Remove District Column

listings = listings.drop(columns=["district"])

print("district column removed")

print("\nCurrent Shape:")
print(listings.shape)

district column removed

Current Shape:
(279712, 32)


In [7]:
# Review Score Missingness Investigation

review_score_columns = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

listings[review_score_columns].isna().sum()

review_scores_rating           91405
review_scores_accuracy         91713
review_scores_cleanliness      91665
review_scores_checkin          91771
review_scores_communication    91687
review_scores_location         91775
review_scores_value            91785
dtype: int64

In [8]:
# Check if Review Score Missingness Occurs Together

review_score_columns = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

missing_all_scores = listings[review_score_columns].isna().all(axis=1)

print("Rows where all review score columns are missing:")
print(missing_all_scores.sum())

print("\nTotal Listings:")
print(len(listings))

Rows where all review score columns are missing:
91377

Total Listings:
279712


In [9]:
# Check if Listings Without Review Scores Exist in Reviews Table

listings_without_scores = listings.loc[
    listings["review_scores_rating"].isna(),
    "listing_id"
]

reviewed_listings = set(reviews["listing_id"])

has_reviews = listings_without_scores.isin(reviewed_listings)

print("Listings without review_scores_rating:")
print(len(listings_without_scores))

print("\nOf those, listings appearing in Reviews table:")
print(has_reviews.sum())

Listings without review_scores_rating:
91405

Of those, listings appearing in Reviews table:
5250


In [10]:
# Review Score Missingness Breakdown

listings_without_scores = set(
    listings.loc[
        listings["review_scores_rating"].isna(),
        "listing_id"
    ]
)

reviewed_listings = set(reviews["listing_id"])

with_reviews_no_scores = len(
    listings_without_scores.intersection(reviewed_listings)
)

without_reviews = len(
    listings_without_scores - reviewed_listings
)

print("Missing Scores + Has Reviews :", with_reviews_no_scores)
print("Missing Scores + No Reviews  :", without_reviews)

Missing Scores + Has Reviews : 5250
Missing Scores + No Reviews  : 86155


In [11]:
# Host Response Fields Missingness Pattern

host_response_columns = [
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate"
]

missing_all_host_response = (
    listings[host_response_columns]
    .isna()
    .all(axis=1)
)

print("Rows where all host response fields are missing:")
print(missing_all_host_response.sum())

print("\nTotal Listings:")
print(len(listings))

Rows where all host response fields are missing:
97169

Total Listings:
279712


In [12]:
# Missingness Pattern Comparison

print("host_response_time missing:")
print(listings["host_response_time"].isna().sum())

print("\nhost_response_rate missing:")
print(listings["host_response_rate"].isna().sum())

print("\nhost_acceptance_rate missing:")
print(listings["host_acceptance_rate"].isna().sum())

host_response_time missing:
128782

host_response_rate missing:
128782

host_acceptance_rate missing:
113087


In [13]:
# Rows where only acceptance rate exists

acceptance_only = listings[
    listings["host_response_time"].isna()
    & listings["host_response_rate"].isna()
    & listings["host_acceptance_rate"].notna()
]

print("Rows with acceptance rate but missing response fields:")
print(len(acceptance_only))

Rows with acceptance rate but missing response fields:
31613


In [14]:
acceptance_only["host_acceptance_rate"].describe()

count    31613.000000
mean         0.789761
std          0.353910
min          0.000000
25%          0.670000
50%          1.000000
75%          1.000000
max          1.000000
Name: host_acceptance_rate, dtype: float64

In [15]:
# Bedrooms Missingness Investigation

print("Missing Bedrooms:")
print(listings["bedrooms"].isna().sum())

print("\nAccommodation Distribution For Missing Bedrooms:")

(
    listings.loc[listings["bedrooms"].isna(), "accommodates"]
            .value_counts()
            .sort_index()
)

Missing Bedrooms:
29435

Accommodation Distribution For Missing Bedrooms:


accommodates
0        85
1      2531
2     19973
3      3238
4      2942
5       265
6       207
7        27
8        41
9         4
10       33
12       11
13        2
14        2
15        7
16       67
Name: count, dtype: int64

In [16]:
# Bedroom Distribution

(
    listings["bedrooms"]
    .value_counts(dropna=False)
    .sort_index()
)

bedrooms
1.0     170163
2.0      51382
3.0      18525
4.0       6579
5.0       2106
6.0        701
7.0        246
8.0        124
9.0         79
10.0       150
11.0        32
12.0        38
13.0         7
14.0        15
15.0        12
16.0        16
17.0         3
18.0         4
19.0         3
20.0        22
21.0         2
22.0         5
23.0         4
24.0         4
25.0         1
26.0         1
30.0         4
31.0         2
33.0         3
34.0         1
35.0         2
38.0         2
39.0         2
40.0        10
41.0         1
42.0         1
46.0         2
48.0         1
50.0        22
NaN      29435
Name: count, dtype: int64

In [17]:
# Extreme Bedroom Listings

listings.loc[
    listings["bedrooms"] >= 20,
    [
        "listing_id",
        "city",
        "property_type",
        "room_type",
        "accommodates",
        "bedrooms",
        "price"
    ]
].sort_values(
    by="bedrooms",
    ascending=False
).head(30)

,listing_id,city,property_type,room_type,accommodates,bedrooms,price
44226,16037944,Mexico City,Entire apartment,Entire place,16,50.0,3614
91807,20118571,Istanbul,Private room in serviced apartment,Private room,16,50.0,214
92133,6810808,Mexico City,Room in bed and breakfast,Hotel room,16,50.0,1832
96185,45740152,Mexico City,Room in hotel,Private room,2,50.0,600
95290,43211747,Istanbul,Room in hotel,Private room,2,50.0,521
97667,43711025,Istanbul,Private room in hostel,Private room,16,50.0,280
98314,45221762,Istanbul,Room in aparthotel,Private room,16,50.0,297
98317,45775825,Istanbul,Room in boutique hotel,Private room,3,50.0,250
97294,45514646,Istanbul,Room in hotel,Private room,16,50.0,120
97285,42215661,Istanbul,Room in hotel,Private room,2,50.0,180


In [18]:
(
    listings.loc[listings["bedrooms"].isna()]
            .groupby("property_type")
            .size()
            .sort_values(ascending=False)
            .head(20)
)

property_type
Entire apartment                      20527
Private room in apartment              1868
Entire condominium                     1278
Entire loft                             815
Private room in house                   496
Room in boutique hotel                  495
Entire serviced apartment               492
Entire guest suite                      401
Entire house                            342
Private room in condominium             333
Room in hotel                           325
Room in aparthotel                      259
Private room in bed and breakfast       224
Entire guesthouse                       160
Private room in serviced apartment      151
Private room in townhouse               139
Private room in guest suite             121
Private room in hostel                  104
Room in serviced apartment               99
Private room in guesthouse               93
dtype: int64

In [19]:
# Bedroom Missingness by Room Type

(
    listings.loc[listings["bedrooms"].isna()]
            .groupby("room_type")
            .size()
            .sort_values(ascending=False)
)

room_type
Entire place    24569
Private room     4398
Hotel room        468
dtype: int64

In [20]:
# Low-Missingness Host Fields

low_missing_cols = [
    "host_since",
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "host_total_listings_count"
]

listings[low_missing_cols].isna().sum()

host_since                   165
host_is_superhost            165
host_has_profile_pic         165
host_identity_verified       165
host_total_listings_count    165
dtype: int64

In [21]:
host_profile_missing = (
    listings[
        [
            "host_since",
            "host_is_superhost",
            "host_has_profile_pic",
            "host_identity_verified",
            "host_total_listings_count"
        ]
    ]
    .isna()
    .all(axis=1)
)

print("Rows where all host profile fields are missing:")
print(host_profile_missing.sum())

Rows where all host profile fields are missing:
165


In [22]:
# Investigate Missing Host Profiles

listings.loc[
    host_profile_missing,
    ["listing_id", "host_id"]
].head()

,listing_id,host_id
52879,5872399,1721954
52880,6300239,1721954
52881,6875069,20618245
52882,7070896,37073350
52883,7526817,33200781


In [23]:
# Hosts Affected

missing_host_rows = listings.loc[
    host_profile_missing,
    "host_id"
]

print("Affected Rows:")
print(len(missing_host_rows))

print("\nUnique Hosts:")
print(missing_host_rows.nunique())

Affected Rows:
165

Unique Hosts:
124


In [24]:
# Remove Rows With Missing Host Profile

listings = listings.loc[
    ~host_profile_missing
].copy()

print("Updated Listings Shape:")
print(listings.shape)

Updated Listings Shape:
(279547, 32)


In [25]:
# Create Working Copy

listings_clean = listings.copy()
reviews_clean = reviews.copy()

print("Listings Clean Shape :", listings_clean.shape)
print("Reviews Clean Shape  :", reviews_clean.shape)

Listings Clean Shape : (279547, 32)
Reviews Clean Shape  : (5373143, 4)


### Data Type Standardization

In [26]:
# Convert Date Columns

listings_clean["host_since"] = pd.to_datetime(
    listings_clean["host_since"]
)

reviews_clean["date"] = pd.to_datetime(
    reviews_clean["date"]
)

print(listings_clean["host_since"].dtype)
print(reviews_clean["date"].dtype)

datetime64[us]
datetime64[us]


In [27]:
# Duplicate review_id records

duplicate_review_ids = reviews_clean[
    reviews_clean["review_id"].duplicated(keep=False)
].copy()

print("Duplicate Rows:")
print(len(duplicate_review_ids))

print("\nUnique Duplicate review_ids:")
print(duplicate_review_ids["review_id"].nunique())

Duplicate Rows:
320

Unique Duplicate review_ids:
160


In [28]:
duplicate_review_analysis = (
    reviews_clean.groupby("review_id")["listing_id"]
    .nunique()
)

duplicate_review_analysis = duplicate_review_analysis[
    duplicate_review_analysis > 1
]

print(duplicate_review_analysis.value_counts())

listing_id
2    160
Name: count, dtype: int64


In [29]:
reviews_clean = reviews_clean.drop_duplicates(
    subset="review_id",
    keep="first"
)

print(
    "\nRemaining Duplicate review_ids:",
    reviews_clean["review_id"].duplicated().sum()
)


Remaining Duplicate review_ids: 0


In [30]:
# Review ID Quality Check

duplicate_review_count = (
    reviews_clean["review_id"]
    .duplicated()
    .sum()
)

print(
    f"Remaining duplicate review_ids: {duplicate_review_count}"
)

Remaining duplicate review_ids: 0


### Duplicate Review Treatment

A total of 160 duplicate review records were identified.

Investigation showed that duplicate records shared the same review_id, reviewer_id, and review date while referencing different listing_ids.

Because a review_id should uniquely identify a review, these records were classified as conflicting review assignments and removed.

Records Removed: 160

The treatment restored review_id uniqueness prior to analytical modeling.

In [31]:
# Domain Issues Summary

print("Price <= 0")
print((listings_clean["price"] <= 0).sum())

print("\nAccommodates <= 0")
print((listings_clean["accommodates"] <= 0).sum())

Price <= 0
113

Accommodates <= 0
85


In [32]:
# Listings With Invalid Price Or Accommodates

invalid_listings = listings_clean.loc[
    (listings_clean["price"] <= 0) |
    (listings_clean["accommodates"] <= 0),
    [
        "listing_id",
        "city",
        "property_type",
        "room_type",
        "price",
        "accommodates",
        "bedrooms"
    ]
]

print("Affected Rows:")
print(len(invalid_listings))

invalid_listings.head(20)

Affected Rows:
113


,listing_id,city,property_type,room_type,price,accommodates,bedrooms
98209,44312919,Paris,Room in boutique hotel,Hotel room,0,0,NaN
202022,42830099,New York,Room in boutique hotel,Hotel room,0,0,NaN
202023,43247472,New York,Room in boutique hotel,Hotel room,0,0,NaN
203253,43205598,New York,Room in boutique hotel,Hotel room,0,0,NaN
203254,44567521,New York,Room in boutique hotel,Hotel room,0,0,NaN
203255,45985185,New York,Room in boutique hotel,Hotel room,0,0,NaN
203256,46251446,New York,Room in boutique hotel,Hotel room,0,0,NaN
203257,43012882,Paris,Room in hotel,Hotel room,0,0,NaN
203258,43035744,Paris,Room in boutique hotel,Hotel room,0,0,NaN
203259,43274767,Paris,Room in boutique hotel,Hotel room,0,0,NaN


In [33]:
# Remove Invalid Listings

invalid_mask = (
    (listings_clean["price"] <= 0) |
    (listings_clean["accommodates"] <= 0)
)

print("Rows Removed:")
print(invalid_mask.sum())

listings_clean = listings_clean.loc[
    ~invalid_mask
].copy()

print("\nUpdated Shape:")
print(listings_clean.shape)

Rows Removed:
113

Updated Shape:
(279434, 32)


In [34]:
print("Price <= 0")
print((listings_clean["price"] <= 0).sum())

print("\nAccommodates <= 0")
print((listings_clean["accommodates"] <= 0).sum())

Price <= 0
0

Accommodates <= 0
0


In [35]:
# Extreme Minimum Nights

listings_clean.loc[
    listings_clean["minimum_nights"] > 365,
    [
        "listing_id",
        "city",
        "property_type",
        "room_type",
        "price",
        "minimum_nights"
    ]
].sort_values(
    by="minimum_nights",
    ascending=False
).head(20)

,listing_id,city,property_type,room_type,price,minimum_nights
180165,8179304,Paris,Private room in loft,Private room,50,9999
255298,4204302,New York,Entire apartment,Entire place,180,1250
11492,34028716,Paris,Entire apartment,Entire place,185,1125
89314,20010285,Sydney,Private room in apartment,Private room,14568,1125
91187,36923173,Sydney,Entire house,Entire place,130,1125
5316,24868060,Paris,Entire apartment,Entire place,20,1124
80504,38487619,Cape Town,Entire villa,Entire place,34529,1124
112525,2942732,New York,Entire condominium,Entire place,60,1124
177550,41989834,Sydney,Private room in condominium,Private room,40,1123
89324,5579404,Paris,Entire townhouse,Entire place,300,1112


In [36]:
# Extreme Maximum Nights

listings_clean.loc[
    listings_clean["maximum_nights"] > 3650,
    [
        "listing_id",
        "city",
        "property_type",
        "room_type",
        "maximum_nights"
    ]
].sort_values(
    by="maximum_nights",
    ascending=False
)

,listing_id,city,property_type,room_type,maximum_nights
228967,4234075,Rome,Entire apartment,Entire place,2147483647
251162,6357527,New York,Entire apartment,Entire place,2147483647
259338,744242,Istanbul,Entire apartment,Entire place,2147483647
268454,628044,Rio de Janeiro,Entire apartment,Entire place,999999999
196791,7693424,New York,Private room in apartment,Private room,20000000
252252,6305027,New York,Entire apartment,Entire place,20000000
197754,12501605,Paris,Private room in apartment,Private room,10000000
221581,3208573,Istanbul,Entire apartment,Entire place,10000000
215467,22114085,Cape Town,Private room in house,Private room,10000000
159137,18037006,Istanbul,Room in boutique hotel,Private room,999999


In [37]:
# Very Large Maximum Nights

print("maximum_nights > 10000")
print((listings_clean["maximum_nights"] > 10000).sum())

print("\nmaximum_nights > 100000")
print((listings_clean["maximum_nights"] > 100000).sum())

maximum_nights > 10000
26

maximum_nights > 100000
12


In [38]:
listings_clean["maximum_nights"].value_counts().head(20)

maximum_nights
1125    157495
30       20434
365      15554
90        8161
60        6362
7         5242
15        5150
28        4333
14        4016
10        3997
180       3259
31        3190
20        2922
21        2344
120       2314
5         1928
29        1847
1124      1788
100       1458
6         1330
Name: count, dtype: int64

In [39]:
# Replace Sentinel Maximum Night Values

listings_clean["maximum_nights"] = np.where(
    listings_clean["maximum_nights"] > 1125,
    1125,
    listings_clean["maximum_nights"]
)

print(
    listings_clean["maximum_nights"].max()
)

1125


In [40]:
missing_after_cleaning = (
    listings_clean.isna()
    .sum()
    .reset_index()
)

missing_after_cleaning.columns = [
    "column",
    "missing_count"
]

missing_after_cleaning = missing_after_cleaning[
    missing_after_cleaning["missing_count"] > 0
].sort_values(
    by="missing_count",
    ascending=False
)

print(missing_after_cleaning.to_string(index=False))

                     column  missing_count
         host_response_time         128536
         host_response_rate         128536
       host_acceptance_rate         112857
        review_scores_value          91615
     review_scores_location          91605
      review_scores_checkin          91601
     review_scores_accuracy          91543
review_scores_communication          91517
  review_scores_cleanliness          91495
       review_scores_rating          91235
                   bedrooms          29302
              host_location            675
                       name            174


In [41]:
# Minor Missing Records

print("Missing Name")
print(listings_clean["name"].isna().sum())

print("\nMissing Host Location")
print(listings_clean["host_location"].isna().sum())

print("\nRows Missing Either Field")
print(
    listings_clean[
        listings_clean["name"].isna() |
        listings_clean["host_location"].isna()
    ].shape[0]
)

Missing Name
174

Missing Host Location
675

Rows Missing Either Field
849


In [42]:
# Overlap Check

both_missing = (
    listings_clean["name"].isna() &
    listings_clean["host_location"].isna()
)

print("Rows with both fields missing:")
print(both_missing.sum())

Rows with both fields missing:
0


In [43]:
cleaning_summary = pd.DataFrame({
    "Cleaning Action": [
        "Removed district column",
        "Removed rows with missing host profiles",
        "Converted host_since to datetime",
        "Converted review date to datetime",
        "Removed duplicate review_ids",
        "Removed invalid listings",
        "Standardized maximum_nights values"
    ],
    "Impact": [
        "Dropped New York-specific field",
        "165 rows removed",
        "Datatype standardized",
        "Datatype standardized",
        "160 duplicate review_ids removed",
        "113 rows removed",
        "Capped sentinel values above 1125"
    ]
})

print(cleaning_summary.to_string(index=False))

                        Cleaning Action                            Impact
                Removed district column   Dropped New York-specific field
Removed rows with missing host profiles                  165 rows removed
       Converted host_since to datetime             Datatype standardized
      Converted review date to datetime             Datatype standardized
           Removed duplicate review_ids  160 duplicate review_ids removed
               Removed invalid listings                  113 rows removed
     Standardized maximum_nights values Capped sentinel values above 1125


### Cleaning Conclusion

The dataset was cleaned by removing invalid records, resolving duplicate review assignments, standardizing data types, addressing structural issues, and correcting system-generated anomalies. Missing values carrying business meaning were retained for future analytical interpretation rather than imputed or removed.

The cleaning process restored review_id uniqueness, improved overall data quality, and produced a reliable analytical dataset suitable for modeling, SQL analysis, KPI development, and dashboard reporting.

In [44]:
listings_clean.to_csv(
    "../data/processed/listings_clean.csv",
    index=False
)

reviews_clean.to_csv(
    "../data/processed/reviews_clean.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [45]:
reviews_clean["review_id"].duplicated().sum()

np.int64(0)

In [46]:
reviews_clean["review_id"].duplicated().sum()

np.int64(0)

In [47]:
len(reviews_clean)

5372983